In [ ]:
import pandas as pd
import numpy as np
import os
import scipy
import matplotlib.pyplot as plt

In [ ]:
data=scipy.io.loadmat('/content/cars_meta.mat')

In [ ]:
car_names=data['class_names'][0]
car_names.shape

In [ ]:
for i ,name in enumerate(car_names):
  print(f'ID:{i},name:{name}')

In [ ]:
train_datas=scipy.io.loadmat('/content/cars_train_annos.mat')

In [ ]:
train_data=train_datas['annotations']

In [ ]:
train_data['class'][0][0]
calsses=train_data['class'][0]

In [ ]:
train_data['class'][0][0]

In [ ]:
len(np.unique(calsses))

In [ ]:
for ann in train_data:
   filename=ann['fname'][1]
   classID=ann['class'][1]
   print(filename)
   print(classID)

In [ ]:
!pip install rarfile

In [ ]:
import rarfile

In [ ]:
with rarfile.RarFile('/content/drive/MyDrive/cars_train_3.rar','r') as rar_file:
  images_path=rar_file.extractall('car_train_images')

In [ ]:
folder_car=os.listdir('/content/car_train_images')

In [ ]:
len(folder_car)

In [ ]:
folder='/content/car_train_images'

In [ ]:
records=[]

for ann in train_datas['annotations'][0]:

  filename=ann['fname'][0]
  class_ID=int(ann['class'][0][0])

  image_path=os.path.join(folder,filename)
  records.append({
      'filename':filename,
      'image_path':image_path,
      'label':class_ID
  })

df=pd.DataFrame(records)

In [ ]:
df

In [ ]:
df.info()

In [ ]:
from PIL import Image

In [ ]:
car_name=pd.Series(car_names)

In [ ]:
car_name.index=range(1,len(car_name)+1)

In [ ]:
car_name.tail()

In [ ]:
df['car_name']=df['label'].map(car_name)

In [ ]:
df.info()

In [ ]:
df['car_name']=df['car_name'].apply(lambda x:x.item())

In [ ]:
type(df['car_name'].iloc[1])

In [ ]:
images_to=df['image_path'].iloc[:5]

In [ ]:
df['label']=df['label']-1

In [ ]:
df

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x=df['image_path']
y=df['label']

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42)

In [ ]:
x_train.shape
y_train.shape

In [ ]:
y_train

In [ ]:
import torch
from torch.utils.data import Dataset,DataLoader
from torchvision.transforms import transforms

In [ ]:
transform=transforms.Compose([
    transforms.RandomResizedCrop((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])

])
resnet_transform_test = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
custom_transformer=transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])
custom_transformer_test=transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

In [ ]:
class cars_data(Dataset):
  def __init__(self,x,y,tranforms=None):
    self.x=x
    self.y=y
    self.tranforms=tranforms

  def __len__(self):
    return self.x.shape[0]

  def __getitem__(self, index):
    image=self.x.iloc[index]
    label=self.y.iloc[index]

    actual_image=Image.open(image).convert('RGB')
    if self.tranforms:
      actual_image=self.tranforms(actual_image)

    return actual_image,label


In [ ]:
train_dataset_custom=cars_data(x_train,y_train,custom_transformer)
test_dataset_custom=cars_data(x_test,y_test,custom_transformer_test)

In [ ]:
train_dataset_restnet=cars_data(x_train,y_train,transform)
test_dataset_restnet=cars_data(x_test,y_test,resnet_transform_test)

In [ ]:
len(train_dataset_custom)

In [ ]:
train_loader_custom=DataLoader(train_dataset_custom,batch_size=50)
test_loader_custom=DataLoader(test_dataset_custom,batch_size=50,shuffle=True)

In [ ]:
train_loader_restnet=DataLoader(train_dataset_restnet,batch_size=32)
test_loader_restnet=DataLoader(test_dataset_restnet,batch_size=32,shuffle=True)

In [ ]:
import torch.nn as nn

In [ ]:
class cars_cnn(nn.Module):
  def __init__(self):
    super().__init__()
    self.features=nn.Sequential(
        nn.Conv2d(3,64,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(64,128,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(128,256,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.MaxPool2d((2)),
        nn.Conv2d(256,512,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(512),
        nn.ReLU(),
        nn.AdaptiveMaxPool2d((4,4)))



    self.classifier=nn.Sequential(
        nn.Flatten(),
        nn.Linear(512*4*4,512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512,196)
    )

  def forward(self,x):
      x=self.features(x)
      x=self.classifier(x)

      return x

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('the device using is :',device)

In [ ]:
model=cars_cnn().to(device)
criterion=nn.CrossEntropyLoss()

In [ ]:
epochs=25
optimizer=torch.optim.Adam(model.parameters(),lr=1e-4)

In [ ]:
for epo in range(epochs):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for feat, label in train_loader_custom:
        feat = feat.to(device)
        label = label.to(device)

        optimizer.zero_grad()
        output = model(feat)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = output.argmax(dim=1)
        correct += (preds == label).sum().item()
        total += label.size(0)

    train_loss = total_loss / len(train_loader_custom)
    train_acc = (correct / total)*100
    print(f'Epoch: {epo+1} | Loss: {train_loss:.4f} | Train Accuracy: {train_acc:.4f}')


In [ ]:
correct=0
total=0
model.eval()
with torch.no_grad():
  for feat2,label2 in test_loader_custom:
     feat2=feat2.to(device)
     label2=label2.to(device)
     output=model(feat2)
     prediction=output.argmax(dim=1)
     correct+=(prediction==label2).sum().item()
     total+=label2.size(0)
  accuracy=(correct/total)*100
  print(f'accuracy is:',accuracy)

In [ ]:
from torchvision import models

In [ ]:
rest_model=models.resnet50(weights="IMAGENET1K_V2")

In [ ]:
rest_model.fc=nn.Linear(rest_model.fc.in_features,196)
rest_model.to(device)

In [ ]:
epochs_rest=20
optimizer_rest=torch.optim.Adam(rest_model.parameters(),lr=1e-4)
criterion2=nn.CrossEntropyLoss()

In [ ]:
for epo in range(epochs_rest):
  rest_model.train()
  total_loss,correct,total=0,0,0
  for feat,label in train_loader_restnet:
    feat=feat.to(device)
    label=label.to(device)
    optimizer_rest.zero_grad()
    output=rest_model(feat)
    loss=criterion2(output,label)
    loss.backward()
    optimizer_rest.step()

    total_loss+=loss.item()
    preds=output.argmax(dim=1)
    correct+=(preds==label).sum().item()
    total+=label.size(0)

  train_loss=total_loss/len(train_loader_restnet)
  train_acc=(correct/total)*100

  print(f'the epoch is:{epo+1},loss:{total_loss:4f},train accuracy:{train_acc:2f}')


In [ ]:
correct=0
total=0
rest_model.eval()
with torch.no_grad():
  for feat2,label2 in test_loader_restnet:
     feat2=feat2.to(device)
     label2=label2.to(device)
     output=rest_model(feat2)
     prediction=output.argmax(dim=1)
     correct+=(prediction==label2).sum().item()
     total+=label2.size(0)
  accuracy=(correct/total)*100
  print(f'accuracy is:',accuracy)

**RESTNET50:81.335**
---
# **CNN:2.75**